In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# Overview of the Goal

The system automatically extracts variables (e.g., $V1, V2$) from the very last "Final Answer" mentioned in the model's response. We are auditing cases where:The model's reasoning is suspicious (too long, too many variables).The order of variables might be wrong in "Strict" scenarios.The automated accuracy check differs from previous records.

# Header Information

GT: The correct variables required for the task.

Parsed: What the system extracted from the model's response.

Order Matters: * <mark style="background: yellow;">YES</mark>: The sequence is vital (e.g., if GT is [1, 2], then [2, 1] is False).

<mark style="background: yellow;">NO</mark>: Only the set of numbers matters (e.g., [1, 2] and [2, 1] are both True).

Reason: Why this specific item was flagged for manual review (e.g., "Long Suffix" or "Multi-Vars").

# Navigation & Labeling
True (Green): The model’s conclusion is correct according to the GT.

False (Red): The model’s conclusion is incorrect.

Invalid (Orange): The response is nonsensical, empty, or fails to provide a clear answer.

Back/Next: Navigate between items. Note: Labeled items are moved to the front of the list automatically upon restarting.

# Workflow
Review the GT and check if Order Matters.

Look at the conclusion after the <mark style="background: yellow; border: 2px solid red;">FINAL ANSWER</mark> anchor.

If you are unsure, read the reasoning text, paying attention to the <mark style="background:#ffcf9e;">Orange</mark> and <mark style="background:#b8ffb8;">Green</mark> highlights to see how the model assigned values.

Reach your conclusions and compare the conclusion to the GT.

Click True, False, or Invalid. The system saves automatically and moves to the next unlabeled item.

When finished, click Save & Exit.

In [ ]:
import os
import pickle
import re
import glob
import ipywidgets as widgets
from IPython.display import display, HTML, Javascript, clear_output


def extract_ordered_variables(response_text):
    if not response_text or not isinstance(response_text, str):
        return []
    lower_text = response_text.lower()
    keyword = "final answer:"
    try:
        start_idx = lower_text.rindex(keyword)
        answer_part = response_text[start_idx + len(keyword):]
        matches = re.findall(r"V(\d+)", answer_part, re.IGNORECASE)
        return [int(m) for m in matches][:2]
    except ValueError:
        return []

def is_strict_file(filename):
    strict_patterns = ["(NOT_A)_OR_B", "(NOT_A)_AND_B", "A_OR_(NOT_B)", "A_AND_(NOT_B)"]
    return any(p in filename for p in strict_patterns)


class MismatchViewer:
    def __init__(self, folder_path):
        self.folder_path = folder_path
        self.file_cache = {}

        self.mismatches = self._collect_data(folder_path)

        if not self.mismatches:
            print("No data matching the filter criteria was found!")
            return

        self.current_idx = self._find_first_unlabeled()

        # --- UI ---
        self.prog_label = widgets.Label()
        self.progress = widgets.IntProgress(value=0, min=0, max=len(self.mismatches), layout={'width': '100%'})
        self.info_area = widgets.HTML()
        self.display_area = widgets.HTML(layout=widgets.Layout(
            height='550px', border='1px solid #ccc', padding='10px', overflow='scroll'
        ))

        self.btn_back = widgets.Button(description="Back", icon='arrow-left')
        self.btn_next = widgets.Button(description="Next", icon='arrow-right')
        self.btn_true = widgets.Button(description="True", button_style='success', layout={'width': '80px'})
        self.btn_false = widgets.Button(description="False", button_style='danger', layout={'width': '80px'})
        self.btn_invalid = widgets.Button(description="Invalid", button_style='warning', layout={'width': '80px'})
        self.btn_clear = widgets.Button(description="Clear Label", icon='trash', layout={'width': '100px'})
        self.btn_exit = widgets.Button(description="Save & Exit", button_style='primary', icon='save')

        self.jump_input = widgets.BoundedIntText(value=1, min=1, max=len(self.mismatches), layout={'width': '80px'})
        self.jump_btn = widgets.Button(description="Go", layout={'width': '50px'})

        self.btn_next.on_click(self.on_next)
        self.btn_back.on_click(self.on_back)
        self.jump_btn.on_click(self.on_jump)
        self.btn_exit.on_click(self.on_exit)
        self.btn_true.on_click(lambda b: self.save_label("True"))
        self.btn_false.on_click(lambda b: self.save_label("False"))
        self.btn_invalid.on_click(lambda b: self.save_label("Invalid"))
        self.btn_clear.on_click(lambda b: self.save_label(None))

    def _collect_data(self, folder_path):
        data_to_show = []

        pkl_files = sorted(glob.glob(os.path.join(folder_path, "*.pkl")))

        for file_path in pkl_files:
            filename = os.path.basename(file_path)
            strict_mode = is_strict_file(filename)
            with open(file_path, 'rb') as f:
                data_list = pickle.load(f)
                self.file_cache[file_path] = data_list

            for i, item in enumerate(data_list):
                raw_res = item.get('llm_raw_response', "")
                gt = item.get('ground_truth', [])
                pred = extract_ordered_variables(raw_res)

                should_pull = False
                pull_reason = ""

                suffix = ""
                if "final answer:" in raw_res.lower():
                    parts = re.split(r'final answer:', raw_res, flags=re.IGNORECASE)
                    suffix = parts[-1]

                vars_in_suffix = re.findall(r"V\d+", suffix, re.IGNORECASE)
                unique_vars_count = len(set(vars_in_suffix))

                old_is_correct = item.get('is_correct', None)
                new_is_correct = (list(pred) == list(gt)) if strict_mode else (set(pred) == set(gt))

                if strict_mode:
                    if set(pred) == set(gt) and list(pred) != list(gt):
                        should_pull = True
                        pull_reason = "Order Error (Strict)"
                else:
                    if unique_vars_count > 2:
                        should_pull = True
                        pull_reason = f"Multi-Vars ({unique_vars_count})"
                    elif len(suffix) > 100:
                        should_pull = True
                        pull_reason = f"Long Suffix ({len(suffix)} chars)"
                    elif old_is_correct is not None and new_is_correct != old_is_correct:
                        should_pull = True
                        pull_reason = "Accuracy Logic Diff"

                if should_pull:
                    data_to_show.append({
                        'file_path': file_path,
                        'filename': filename,
                        'item_idx': i,
                        'strict_mode': strict_mode,
                        'gt': gt,
                        'pred': pred,
                        'content': raw_res,
                        'label': item.get('label', None),
                        'reason': pull_reason,
                        'suffix_len': len(suffix)
                    })

        # --- Modification 2: Final sorting of results ---
        # Sorting priority:
        # 1. Labeled items first (item['label'] is None is False, False < True, so it's sorted first)
        # 2. Filename (ensures data from different files is not interleaved)
        # 3. Intra-file index (ensures the order within the file is fixed)
        data_to_show.sort(key=lambda x: (x['label'] is None, x['filename'], x['item_idx']))

        return data_to_show

    def _find_first_unlabeled(self):
        for idx, item in enumerate(self.mismatches):
            if item['label'] is None: return idx
        return 0

    def highlight_content(self, text, gt):
        html = text.replace("<", "&lt;").replace(">", "&gt;")

        # 2.  (A is/=/as 与 B is/=/as)
        # A is, A =, A as, as A (B 同理)
        p_a = r'\b(A\s*is|A\s*=|A\s*as|as\s*A)\b'
        p_b = r'\b(B\s*is|B\s*=|B\s*as|as\s*B)\b'

        matches_a = list(re.finditer(p_a, html, re.IGNORECASE))
        matches_b = list(re.finditer(p_b, html, re.IGNORECASE))

        spans_to_mark = set()
        for ma in matches_a:
            for mb in matches_b:
                if ma.start() < mb.start():
                    dist = mb.start() - ma.end()
                else:
                    dist = ma.start() - mb.end()

                if 0 <= dist <= 10:
                    spans_to_mark.add(ma.span())
                    spans_to_mark.add(mb.span())

        sorted_spans = sorted(list(spans_to_mark), key=lambda x: x[0], reverse=True)
        for start, end in sorted_spans:
            html = html[:start] + f'<mark style="background:#ffcf9e;">{html[start:end]}</mark>' + html[end:]

        if len(gt) >= 2:
            v1, v2 = f"V{gt[0]}", f"V{gt[1]}"
            pattern = rf'({v1}.{{0,15}}{v2}|{v2}.{{0,15}}{v1})'
            html = re.sub(pattern, r'<mark style="background:#b8ffb8; border-bottom:2px solid green;">\1</mark>', html, flags=re.IGNORECASE)

        lower_html = html.lower()
        target = "final answer:"
        if target in lower_html:
            last_pos = lower_html.rfind(target)
            marker = '<mark id="final-answer-anchor" style="background: yellow; font-weight:bold; border: 2px solid red;">FINAL ANSWER:</mark>'
            html = html[:last_pos] + marker + html[last_pos + len(target):]

        return html

    def refresh_ui(self):
        if not self.mismatches: return
        item = self.mismatches[self.current_idx]
        current_label = item['label']
        label_display = f"<span style='color:gray;'>UNLABELED</span>" if current_label is None else f"<b style='color:purple; background:#f0e6ff; padding:2px 5px;'>{current_label}</b>"

        order_status = "YES" if item['strict_mode'] else "NO"
        self.info_area.value = f"""
        <div style="padding:10px; background-color:#f8f9fa; border-left:5px solid #2196F3; line-height:1.4;">
            <div style="display: flex; justify-content: space-between;">
                <span><b>GT:</b> <code style="color:blue; font-size:16px;">{item['gt']}</code> | <b>Parsed:</b> <code>{item['pred']}</code></span>
                <span>Order Matters: <mark style="background: yellow; padding: 0 4px; font-weight: bold;">{order_status}</mark></span>
            </div>
            <div style="margin-top:5px;">
                <span>Reason: <mark>{item['reason']}</mark></span> |
                <span>Status: {label_display}</span> |
                <span>File: <small>{item['filename']}</small></span>
            </div>
        </div>
        """

        highlighted_text = self.highlight_content(item['content'], item['gt'])
        self.display_area.value = f"<div style='white-space: pre-wrap; font-family: monospace;'>{highlighted_text}</div>"

        self.progress.value = self.current_idx + 1
        self.prog_label.value = f"Data {self.current_idx + 1} / {len(self.mismatches)}"
        self.jump_input.value = self.current_idx + 1

        display(Javascript("""
            setTimeout(function() {
                var el = document.getElementById('final-answer-anchor');
                if (el) { el.scrollIntoView({behavior: "smooth", block: "center"}); }
            }, 150);
        """))

    def save_label(self, label_value):
        item = self.mismatches[self.current_idx]
        file_path = item['file_path']
        idx = item['item_idx']

        self.file_cache[file_path][idx]['label'] = label_value
        item['label'] = label_value

        with open(file_path, 'wb') as f:
            pickle.dump(self.file_cache[file_path], f)

        if label_value is not None:
            self.on_next(None)
        else:
            self.refresh_ui()

    def on_exit(self, b):
        clear_output()
        print("Work Progress Saved. Exit Successful.")

    def on_next(self, b):
        if self.current_idx < len(self.mismatches) - 1:
            self.current_idx += 1
            self.refresh_ui()

    def on_back(self, b):
        if self.current_idx > 0:
            self.current_idx -= 1
            self.refresh_ui()

    def on_jump(self, b):
        target = self.jump_input.value - 1
        if 0 <= target < len(self.mismatches):
            self.current_idx = target
            self.refresh_ui()

    def run(self):
        if self.mismatches:
            label_box = widgets.HBox([
                widgets.Label("🏷️ LABEL:"), self.btn_true, self.btn_false, self.btn_invalid,
                widgets.Label(" | "), self.btn_clear,
                widgets.Label("      "), self.btn_exit
            ])
            nav_box = widgets.HBox([self.btn_back, self.btn_next, widgets.Label("Jump:"), self.jump_input, self.jump_btn])

            ui = widgets.VBox([
                widgets.HBox([self.prog_label, self.progress]),
                self.info_area,
                self.display_area,
                label_box,
                nav_box
            ])
            display(ui)
            self.refresh_ui()

# ================= 运行 =================
PATH = "/content/drive/MyDrive/inference-time-scale-adaptation/data_label/qwen3-instruct"
viewer = MismatchViewer(PATH)
viewer.run()

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

... (69 outputs left out)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import os
import pickle

def convert_label(label):
    if isinstance(label, str):
        if label.lower() == "true":
            return True
        elif label.lower() == "false":
            return False
        else:
            return None
    elif label in [True, False]:
        return label
    return None

def update_is_correct_from_label(input_dir, output_dir):
    os.makedirs(output_dir, exist_ok=True)
    pkl_files = [f for f in os.listdir(input_dir) if f.endswith('.pkl')]

    for fname in pkl_files:
        input_path = os.path.join(input_dir, fname)
        output_path = os.path.join(output_dir, fname)

        try:
            with open(input_path, 'rb') as f:
                data = pickle.load(f)

            # Check: should be a list of dicts
            if not isinstance(data, list) or not all(isinstance(d, dict) for d in data):
                print(f"⚠️ Skipping {fname}: not a list of dicts")
                continue

            modified = False
            for item in data:
                if 'label' in item:
                    label_value = convert_label(item['label'])
                    if label_value is not None:
                        if item.get('is_correct') != label_value:
                            item['is_correct'] = label_value
                            modified = True

            with open(output_path, 'wb') as f:
                pickle.dump(data, f)

            print(f"✅ {fname} ({'modified' if modified else 'no change'}) saved to {output_path}")

        except Exception as e:
            print(f"❌ Error processing {fname}: {e}")

original_folder = "/content/drive/MyDrive/inference-time-scale-adaptation/data_label/deepseek-r1"
output_folder = "/content/drive/MyDrive/inference-time-scale-adaptation/model_trajectory/deepseek-r1"
update_is_correct_from_label(original_folder, output_folder)

original_folder = "/content/drive/MyDrive/inference-time-scale-adaptation/data_label/deepseek-v3"
output_folder = "/content/drive/MyDrive/inference-time-scale-adaptation/model_trajectory/deepseek-v3"
update_is_correct_from_label(original_folder, output_folder)

original_folder = "/content/drive/MyDrive/inference-time-scale-adaptation/data_label/qwen3-instruct"
output_folder = "/content/drive/MyDrive/inference-time-scale-adaptation/model_trajectory/qwen3-instruct"
update_is_correct_from_label(original_folder, output_folder)

original_folder = "/content/drive/MyDrive/inference-time-scale-adaptation/data_label/qwen3-thinking"
output_folder = "/content/drive/MyDrive/inference-time-scale-adaptation/model_trajectory/qwen3-thinking"
update_is_correct_from_label(original_folder, output_folder)



✅ 2_vars_adaptive_sample_5_(NOT_A)_AND_(NOT_B)_DeepSeek-R1.pkl (no change) saved to /content/drive/MyDrive/infer_scale_reason/cogsci_results/deepseek-r1/2_vars_adaptive_sample_5_(NOT_A)_AND_(NOT_B)_DeepSeek-R1.pkl
✅ 2_vars_adaptive_sample_5_A_OR_B_DeepSeek-R1.pkl (no change) saved to /content/drive/MyDrive/infer_scale_reason/cogsci_results/deepseek-r1/2_vars_adaptive_sample_5_A_OR_B_DeepSeek-R1.pkl
✅ 2_vars_adaptive_sample_5_A_XNOR_B_DeepSeek-R1.pkl (no change) saved to /content/drive/MyDrive/infer_scale_reason/cogsci_results/deepseek-r1/2_vars_adaptive_sample_5_A_XNOR_B_DeepSeek-R1.pkl
✅ 2_vars_adaptive_sample_5_NOT_(A_AND_B)_DeepSeek-R1.pkl (modified) saved to /content/drive/MyDrive/infer_scale_reason/cogsci_results/deepseek-r1/2_vars_adaptive_sample_5_NOT_(A_AND_B)_DeepSeek-R1.pkl
✅ 2_vars_adaptive_sample_5_A_AND_B_DeepSeek-R1.pkl (modified) saved to /content/drive/MyDrive/infer_scale_reason/cogsci_results/deepseek-r1/2_vars_adaptive_sample_5_A_AND_B_DeepSeek-R1.pkl
✅ 2_vars_adaptiv